In [ ]:
from pathlib import Path
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema

from setup_utils import create_minio_spark_session, DEV_LOCATION_ID_LIST

### Setup

In [ ]:
spark = create_minio_spark_session()

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# configure download
ev.download.configure(
    api_key='your_key_here'
)

### Pull domain tables

In [ ]:
print("Loading domain tables.")

print("Loading configurations.")
ev.download.configurations(
    load=True
)

print("Loading units.")
ev.download.units(
    load=True
)

print("Loading variables.")
ev.download.variables(
    load=True
)

print("Loading locations.")
ev.download.locations(
    ids=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading location croswalks.")
ev.download.location_crosswalks(
    primary_location_id=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading location attributes.")
ev.download.attributes(
    load=True
)
ev.download.location_attributes(
    location_id=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading domain variables, locations, crosswalks, and attributes complete!")

### Kill spark

In [ ]:
spark.stop()